# FYS4480/FYS9480 -- Week 43: Kohn-Sham and the local density approximation

Companion notebook to the week-43 slides (lecture notes chapter 8, sections
8.5-8.8).  Plain NumPy, self-contained; the same calculations as
`BookPrograms/chapter08/dft.py`, written out from scratch so that every
number on the slides can be checked.

1. The local exchange functional of the three-dimensional gas against
   $-0.916/r_s$, and the factor $4/3$ in $v_x$.
2. The exchange hole $g(y)$ of the Fermi sea and its sum rule, satisfied
   slowly by a $y^{-2}$ tail.
3. A one-dimensional exchange functional built numerically from
   $\rho^{(1)}(s)=\sin(k_Fs)/\pi s$ for the softened Coulomb interaction.
4. Kohn-Sham, Hartree-Fock and exact diagonalisation for $N$ spinless
   fermions in the harmonic trap, in a basis of eight oscillator orbitals;
   the self-interaction error; how local is too local.

In [1]:
import numpy as np
from itertools import combinations
from numpy.polynomial.hermite import hermval
from math import factorial
np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. Exchange in the local density approximation

$\epsilon_x(n) = -\tfrac34(3/\pi)^{1/3} n^{1/3}$ in Hartree atomic units,
with $n = 3/(4\pi r_s^3)$; in Rydberg this must be $-0.916/r_s$.

In [2]:
def eps_x_3d(n):
    return -0.75 * (3 / np.pi)**(1 / 3) * n**(1 / 3)

print("  r_s     eps_x [Ha]   eps_x [Ry]   -0.916/r_s   v_x/eps_x")
for rs in (1.0, 2.0, 4.0, 4.8253, 6.0):
    n = 3 / (4 * np.pi * rs**3)
    ex = eps_x_3d(n)
    h = 1e-6 * n
    vx = ((n + h) * eps_x_3d(n + h) - (n - h) * eps_x_3d(n - h)) / (2 * h)
    print("  %6.4f   %+.6f   %+.6f   %+.6f    %.4f" % (rs, ex, 2 * ex, -0.916 / rs, vx / ex))
print("\ncoefficient: %.6f  (chapter 6: 0.9163)" % (-2 * eps_x_3d(3 / (4 * np.pi))))

  r_s     eps_x [Ha]   eps_x [Ry]   -0.916/r_s   v_x/eps_x
  1.0000   -0.458165   -0.916331   -0.916000    1.3333
  2.0000   -0.229083   -0.458165   -0.458000    1.3333
  4.0000   -0.114541   -0.229083   -0.229000    1.3333
  4.8253   -0.094951   -0.189901   -0.189833    1.3333
  6.0000   -0.076361   -0.152722   -0.152667    1.3333

coefficient: 0.916331  (chapter 6: 0.9163)


## 2. The exchange hole and its sum rule

$g(y) = 1 - \tfrac92\left[(\sin y - y\cos y)/y^3\right]^2$ with $y=k_Fs$;
$\frac{4}{3\pi}\int_0^\infty y^2[g(y)-1]\,dy = -1$.

In [3]:
def g_hole(y):
    y = np.asarray(y, dtype=float)
    out = np.full_like(y, 0.5)
    m = y > 1e-8
    out[m] = 1 - 4.5 * ((np.sin(y[m]) - y[m] * np.cos(y[m])) / y[m]**3)**2
    return out

print("  k_F s     g")
for y in (0.0, 0.5, 1.0, 2.0, 4.0, 8.0):
    print("  %5.2f   %.6f" % (y, g_hole(np.array([y]))[0]))

for ymax in (4, 20, 100, 400, 4000):
    y = np.linspace(0, ymax, int(ymax * 400) + 1)
    integrand = y**2 * (g_hole(y) - 1)
    I = 4 / (3 * np.pi) * np.trapezoid(integrand, y)
    print("sum rule integrated to k_F s = %5d:  %.4f" % (ymax, I))
y = np.linspace(0, 20, 8001)
f = y**2 * (g_hole(y) - 1)
print("principal minimum of y^2[g-1]: %.3f at y = %.2f" % (f.min(), y[f.argmin()]))

  k_F s     g
   0.00   0.500000
   0.50   0.524471
   1.00   0.591838
   2.00   0.786732
   4.00   0.996208
   8.00   0.999920
sum rule integrated to k_F s =     4:  -0.7855
sum rule integrated to k_F s =    20:  -0.9531
sum rule integrated to k_F s =   100:  -0.9904
sum rule integrated to k_F s =   400:  -0.9976
sum rule integrated to k_F s =  4000:  -0.9998
principal minimum of y^2[g-1]: -0.856 at y = 2.08


## 3. A one-dimensional exchange functional, built numerically

Spinless fermions in one dimension with $v(s) = 1/\sqrt{s^2 + a^2}$,
$a = 0.5$: $k_F = \pi n$ and
$\epsilon_x(n) = -\frac{1}{n}\int_0^\infty v(s)\,[\sin(k_Fs)/\pi s]^2\,ds$.
At high density the squared density matrix becomes a spike at $s=0$ and
$\epsilon_x \to -v(0)/2 = -1/(2a) = -1$.

In [4]:
A_SOFT = 0.5
def v_soft(s):
    return 1 / np.sqrt(s**2 + A_SOFT**2)

def eps_x_1d(n, smax=400.0, ns=400001):
    s = (np.arange(ns) + 0.5) * smax / ns
    kF = np.pi * n
    rho1 = np.sin(kF * s) / (np.pi * s)
    return -np.sum(v_soft(s) * rho1**2) * (smax / ns) / n

n_grid = np.linspace(0.005, 3.0, 600)
ex_grid = np.array([eps_x_1d(n) for n in n_grid])
# spline-free interpolation: linear in n for eps_x, finite-difference for v_x
def eps_x_tab(n):
    return np.interp(n, n_grid, ex_grid)
def v_x_tab(n):
    h = 1e-4
    nn = np.maximum(n, h)
    return ((nn + h) * eps_x_tab(nn + h) - (nn - h) * eps_x_tab(nn - h)) / (2 * h)

print("    n     eps_x(n)      v_x(n)")
for n in (0.05, 0.10, 0.20, 0.40, 0.80, 1.50, 2.50):
    print("  %4.2f   %+.8f   %+.8f" % (n, eps_x_1d(n), v_x_tab(np.array([n]))[0]))
print("\nhigh-density limit -v(0)/2 = %.1f" % (-v_soft(0.0) / 2))

    n     eps_x(n)      v_x(n)
  0.05   -0.17352615   -0.29745226
  0.10   -0.27855522   -0.45910447
  0.20   -0.42332628   -0.65791171
  0.40   -0.59557914   -0.84879622
  0.80   -0.75884371   -0.96588047
  1.50   -0.86548980   -0.99703632
  2.50   -0.91895548   -0.99989719

high-density limit -v(0)/2 = -1.0


## 4. Kohn-Sham, Hartree-Fock and exact, on one Hamiltonian

Eight oscillator orbitals ($\hbar=m=\omega=1$), $h_0 = \mathrm{diag}(k+\tfrac12)$,
two-body matrix elements $\langle pq|v|rs\rangle$ computed on a grid.
Hartree-Fock as in week 38; Kohn-Sham with the Hartree term from the same
integrals and the exchange potential $v_x(n(x))$ of part 3 evaluated on the
grid; exact diagonalisation for $N=2$ in the same basis.

In [5]:
M = 8
x = np.linspace(-8, 8, 801)
dx = x[1] - x[0]
phi = np.array([hermval(x, np.eye(M)[k]) * np.exp(-x**2 / 2) / np.sqrt(2.0**k * factorial(k) * np.sqrt(np.pi)) for k in range(M)])
h0 = np.diag(np.arange(M) + 0.5)
Wgrid = v_soft(x[:, None] - x[None, :])
# <pq|v|rs> = int int phi_p(x1) phi_q(x2) v(x1-x2) phi_r(x1) phi_s(x2)
V = np.einsum('pi,ri,ij,qj,sj->pqrs', phi, phi, Wgrid, phi, phi, optimize=True) * dx**2
print("two-body integrals computed; <00|v|00> = %.6f" % V[0, 0, 0, 0])

def density_on_grid(C, N):
    occ = C[:, :N]
    psi = occ.T @ phi                      # (N, grid)
    return (psi**2).sum(axis=0)

def hartree_fock(N, iters=500, tol=1e-12, mix=0.5):
    C = np.eye(M); E_old = 0
    for it in range(iters):
        rho = C[:, :N] @ C[:, :N].T
        f = h0 + np.einsum('prqs,rs->pq', V, rho) - np.einsum('prsq,rs->pq', V, rho)
        e, C_new = np.linalg.eigh(f)
        C = C_new
        E = np.trace(rho @ h0) + 0.5 * np.einsum('pq,prqs,rs->', rho, V, rho) - 0.5 * np.einsum('pq,prsq,rs->', rho, V, rho)
        if abs(E - E_old) < tol: break
        E_old = E
    return E, C, it

def kohn_sham(N, iters=2000, tol=1e-11, mix=0.3):
    C = np.eye(M); rho = C[:, :N] @ C[:, :N].T; E_old = 0
    for it in range(iters):
        n = density_on_grid(C, N)
        vx = v_x_tab(n)
        Vx = np.einsum('pi,i,qi->pq', phi, vx, phi) * dx
        J = np.einsum('prqs,rs->pq', V, rho)
        e, C = np.linalg.eigh(h0 + J + Vx)
        rho_new = C[:, :N] @ C[:, :N].T
        rho = mix * rho_new + (1 - mix) * rho
        n = density_on_grid(C, N)
        E_H = 0.5 * np.einsum('pq,prqs,rs->', rho, V, rho)
        E_x = np.sum(n * eps_x_tab(n)) * dx
        T_ext = np.trace(rho @ h0)
        E = T_ext + E_H + E_x
        if abs(E - E_old) < tol and it > 5: break
        E_old = E
    return E, C, it, dict(E_H=E_H, E_x=E_x)

def exact_energy(N):
    dets = [sum(1 << p for p in c) for c in combinations(range(M), N)]
    idx = {d: k for k, d in enumerate(dets)}
    def apply(det, ops):
        sign = 1
        for p, dag in reversed(ops):
            if dag == ((det >> p) & 1): return None
            sign *= (-1)**bin(det & ((1 << p) - 1)).count('1'); det ^= 1 << p
        return sign, det
    H = np.zeros((len(dets), len(dets)))
    for k, d in enumerate(dets):
        for p in range(M):
            r = apply(d, [(p, 1), (p, 0)])
            if r: H[idx[r[1]], k] += r[0] * h0[p, p]
        for p in range(M):
            for q in range(M):
                for r_ in range(M):
                    for s in range(M):
                        r = apply(d, [(p, 1), (q, 1), (s, 0), (r_, 0)])
                        if r: H[idx[r[1]], k] += 0.5 * r[0] * V[p, q, r_, s]
    return np.linalg.eigvalsh(H)[0]

print("  N   E(Kohn-Sham)   E(Hartree-Fock)   E(exact)")
results = {}
for N in (1, 2, 3, 4):
    E_ks, C_ks, it_ks, parts = kohn_sham(N)
    E_hf, C_hf, it_hf = hartree_fock(N)
    E_ex = exact_energy(N) if N <= 2 else float('nan')
    results[N] = (E_ks, C_ks, parts, E_hf)
    print("  %d   %.8f     %.8f        %s" % (N, E_ks, E_hf, "%.8f" % E_ex if N <= 2 else "   --"))
print("\n(chapter 8, Table 8.1: 0.54752674 / 0.50000000, 2.72022768 / 2.66308878 / 2.65923049, 6.45108052 / 6.39016133, 11.68602229 / 11.62305385)")

two-body integrals computed; <00|v|00> = 1.228286
  N   E(Kohn-Sham)   E(Hartree-Fock)   E(exact)
  1   0.54751809     0.50000000        0.50000000
  2   2.72022223     2.66308878        2.65923049
  3   6.45107618     6.39016133           --
  4   11.68601813     11.62305385           --

(chapter 8, Table 8.1: 0.54752674 / 0.50000000, 2.72022768 / 2.66308878 / 2.65923049, 6.45108052 / 6.39016133, 11.68602229 / 11.62305385)


### The self-interaction error

For one particle the Hartree energy and the exchange energy must cancel
exactly.  A determinant does this term by term; a local functional cannot.

In [6]:
E_ks1, C1, parts1, _ = results[1]
print("N = 1:  U_H = %.8f   E_x^LDA = %.8f   sum = %.8f  (should be 0)" % (parts1['E_H'], parts1['E_x'], parts1['E_H'] + parts1['E_x']))
print("        Kohn-Sham misses the exact 0.5 by %.4f" % (E_ks1 - 0.5))

N = 1:  U_H = 0.60855052   E_x^LDA = -0.56166818   sum = 0.04688233  (should be 0)
        Kohn-Sham misses the exact 0.5 by 0.0475


### How local is too local?

On one and the same Kohn-Sham density, compare the exact exchange energy of
the determinant, $-\tfrac12\sum_{kl}\langle kl|v|lk\rangle$, with the local
estimate $\int n\,\epsilon_x(n)\,dx$.

In [7]:
print("  N    E_x exact    E_x local    ratio    N(1-r)")
for N in (1, 2, 3, 4):
    E_ks, C, parts, _ = results[N]
    rho = C[:, :N] @ C[:, :N].T
    Ex_exact = -0.5 * np.einsum('pq,prsq,rs->', rho, V, rho)
    r = parts['E_x'] / Ex_exact
    print("  %d   %+.8f   %+.8f   %.4f   %.3f" % (N, Ex_exact, parts['E_x'], r, N * (1 - r)))
print("\nThe shortfall scales as ~0.078/N: an error that lives in the edges of the density, not in its bulk.")

  N    E_x exact    E_x local    ratio    N(1-r)
  1   -0.60855052   -0.56166818   0.9230   0.077
  2   -1.33076913   -1.27749360   0.9600   0.080
  3   -2.11085909   -2.05547577   0.9738   0.079
  4   -2.92523548   -2.86867162   0.9807   0.077

The shortfall scales as ~0.078/N: an error that lives in the edges of the density, not in its bulk.


### The densities

Four maxima for four particles in every treatment: the shell structure is
back.  Hartree-Fock is the more strongly modulated; Kohn-Sham is a little
more diffuse -- an electron spreading out to escape its own charge.

In [8]:
N = 4
n_ks = density_on_grid(results[N][1], N)
n_hf = density_on_grid(hartree_fock(N)[1], N)
n_0 = density_on_grid(np.eye(M), N)
def maxima(n):
    m = (n[1:-1] > n[:-2]) & (n[1:-1] > n[2:])
    return x[1:-1][m], n[1:-1][m]
for label, n in (("non-interacting", n_0), ("Hartree-Fock", n_hf), ("Kohn-Sham", n_ks)):
    xm, nm = maxima(n)
    print("%-16s maxima at %s, heights %s, n(0) = %.3f, weight beyond |x|=3: %.3f"
          % (label, np.round(xm, 2), np.round(nm, 3), n[np.argmin(np.abs(x))], np.sum(n[np.abs(x) > 3]) * dx))
print("max |n_KS - n_HF| = %.3f" % np.abs(n_ks - n_hf).max())

non-interacting  maxima at [-1.66 -0.52  0.52  1.66], heights [0.806 0.943 0.943 0.806], n(0) = 0.846, weight beyond |x|=3: 0.025
Hartree-Fock     maxima at [-1.88 -0.6   0.6   1.88], heights [0.77  0.884 0.884 0.77 ], n(0) = 0.735, weight beyond |x|=3: 0.060
Kohn-Sham        maxima at [-1.86 -0.58  0.58  1.86], heights [0.751 0.858 0.858 0.751], n(0) = 0.761, weight beyond |x|=3: 0.064
max |n_KS - n_HF| = 0.028
